# Start

In [28]:
# Taruh ini di sel paling atas sendiri di Notebook-mu
%load_ext autoreload
%autoreload 2

## sensor_simulation.py

In [1]:
%%writefile sensor_simulation.py
from datetime import datetime, timezone, timedelta
import random

def sensor_simulation() -> dict[str,float]:
  tz_wib = timezone(timedelta(hours=7))
  temperature = random.uniform(20.0, 40.0)
  air_humidity = random.uniform(40.0, 100.0)
  soil_moisture = random.uniform(0.0, 100.0)
  soil_ph = random.uniform(4.0, 7.0)

  return {
      'reading_timestamp': datetime.now(tz_wib).isoformat(),
      'temperature': temperature,
      'air_humidity': air_humidity,
      'soil_moisture': soil_moisture,
      'soil_ph': soil_ph

      ## if prefer lower decimal count to save space
      # round('temperature': temperature, 3),
      # round('air_humidity': air_humidity, 3),
      # round('soil_moisture': soil_moisture, 3),
      # round('soil_ph': soil_ph, 3)
  }

Writing sensor_simulation.py


In [2]:
from sensor_simulation import sensor_simulation

print(sensor_simulation())

{'reading_timestamp': '2026-05-24T14:56:00.140646+07:00', 'temperature': 20.976981456895622, 'air_humidity': 59.927983217142405, 'soil_moisture': 89.36187265339922, 'soil_ph': 4.101401025839015}


In [4]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print(AESGCM.generate_key(bit_length=256))

b'\x16S\xec\x1f\x19\xde\x1e\x92\x0e\xb7\x8da\xd9\x10\xbdp\x94\xc3\x83\xd3\xae\xe9\x0c\xffte\xf1N\x95\x0f\xecf'


## rsa_encryption.py

In [44]:
%%writefile rsa_encryption.py
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import hashes

def rsa_encrypt(message: bytes, public_key_pem: str) -> bytes:
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    ciphertext = public_key.encrypt(
        message,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return ciphertext

Overwriting rsa_encryption.py


## build_payload.py

In [55]:
%%writefile build_payload.py
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from rsa_encryption import rsa_encrypt
from sensor_simulation import sensor_simulation

# def get_session_key() -> bytes | None:
#   if session_key_use_count > session_key_lifetime:
#	 return AESGCM.generate_key(bit_length=256)
#   else:
#	 return None



def build_payload(public_key_pem: str) -> dict:
	tz_wib = timezone(timedelta(hours=7))

	# this one gonna cascade depending on the result. None is not a valid session_key. You need to use the current session_key
	# session_key = get_session_key() # -> raw_bytes
	# session_key for testing because get_session_key isn't finished yet
	session_key = b'\xf8\xb1\xab)4\xac\xdf\xfa\x8b)\xbf\xd9\xd9^c\xc5\x17\x11\x8f\xfe{\xeb\x00u\xaa9\xac:\x8aVp\xcd'
	encrypted_session_key = rsa_encrypt(session_key, public_key_pem) # -> raw_bytes

	aesgcm = AESGCM(session_key)
	iv = os.urandom(12)
	plaintext_bytes = json.dumps(sensor_simulation()).encode('utf-8') # -> dict[str,float]
	aad = {
		'sensor_id': 1,
		'transmission_timestamp': datetime.now(tz_wib).isoformat(),
		'encrypted_session_key': base64.b64encode(encrypted_session_key).decode('utf-8'),
	}
	aad_bytes = json.dumps(aad).encode('utf-8')

	encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
	ciphertext = encrypted_raw[:-16]
	tag = encrypted_raw[-16:]

	return {
		**aad,
		'nonce': base64.b64encode(iv).decode('utf-8'),
		'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
		'tag': base64.b64encode(tag).decode('utf-8')
	}

Overwriting build_payload.py


## get_public_key.py

In [67]:
%%writefile get_public_key.py
import requests
from build_payload import build_payload
# import json

def get_public_key() -> str:
    url='http://localhost:3000/public-key'
    response = requests.get(url)
    response.raise_for_status()
    return response.json()['public_key']
    # return response

# server_public_key = get_server_public_key()
# print(server_public_key)

Writing get_public_key.py


In [41]:
from cryptography.hazmat.primitives.serialization import load_pem_public_key

# 1. String public key mentah (misal hasil ambil dari Node.js kemarin)
public_key_pem = server_public_key

# 2. Import fungsinya dan ubah string menjadi objek objek kunci beneran
public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

# Sekarang variabel 'public_key' sudah menjadi objek RSA asli 
# yang siap digunakan untuk enkripsi data!
print(public_key)
# Output: <class 'cryptography.hazmat.backends.openssl.rsa._RSAPublicKey'>

## Payload content

In [49]:
from payload import build_payload

print(build_payload(server_public_key))

{'sensor_id': 1, 'transmission_timestamp': '2026-05-24T22:05:21.180305+07:00', 'encrypted_session_key': 'effp6qXeQfPQSMXwbDtZ6sf/BQFow8K774khSG3YdSDsSJMAIMk9oxHhvaZv3Xe7l5sA1DqrjBFl8yIm/tJS0v51Pd4HkbqRfzuJ0sFJ6sPPVac9RHE4bF7nGPeI7npvt/9OVZpOSI8k7F4LKO6VpOxANoui6vmwVytbSxMjfG1u14i0gE+1zlae9wshHaIBoEaIA/F9fptwvpBikgLKzjK3Nahdbb8iOUxordA1l5rW+6E+nGlge6my017dTktytrlsikMEzVXSPKFElMWyLazVkNkBOBeSVBg2mdvx2g4VDuYlYsd02UoDIqhi+7IJoy7mr73eHIiTNHbBevgtNQ==', 'nonce': 'HJrE+O6TeSd0w1T0', 'ciphertext': '6v8mx4zaBI/5Lqad4iSQS/A5Kfdh6sSPOTWJ5U0/ozX47zahm+vTlbECAijeUPsjn2bfTF8ePb7usEe12vUYxPiUZ11wBDdJsmX3V50ygcWxvWo5iKE2VGYoUDsDSxJg6baItNCwFogN3W/UlDmuSor55QFXkm5aLTa2FMwuCz9UDNW3l+WnRlDF7vkK3p4AL2KtupppvE78ufWnyQm2pnXlOkxRJnmDuBmE2WHW10nHLbBV3TNXYLkwDru20Fpe', 'tag': 'MD50gR5x68rOq7cnZlJPQA=='}


## Send payload

In [69]:
import requests
from build_payload import build_payload
from get_public_key import get_public_key

server_public_key = get_public_key()

url = 'http://localhost:3000/telemetry'
payload = build_payload(server_public_key)
print('payload: ', payload)
response=requests.post(url, json=payload)
print(response, response.json())

payload:  {'sensor_id': 1, 'transmission_timestamp': '2026-05-26T01:29:23.467066+07:00', 'encrypted_session_key': 'OfGlqL0NxvtKtkpNsVpZkuofeJaZDF5f+WDXv7xTlAZ+391cJnpn/LnlLv9IyReM2w/KbD/WeUOix/Mf6JmN/IqFsj5rI+tPuZ+u5RXEsiQe/Jkxif+I/uXlYCXsUyqeuewJiXJK6sD8+bmFCAckn2orajhlLkmGOOcIJXl0AtCtmzcUiteF3Gfnt1wjnytzGwXAIvFUU2odEO4P/g5c+HbcTtnzeJTj4y+/bFvxnov0YdL1YPHJicmKrDvh7Xqk5/qVNLJVCj6YpbVeBnMFDDWxlgVbeyGTGsvCyzThPzcePmh+5FY4Citvf3GhRs5H178AHpd4FBHPfTO3yLcu1g==', 'nonce': 'dE07LKyVWW9lcg6p', 'ciphertext': 'geI5c1BZ2YCejFsQ6DJ3Sl/xt9vEjQWdCDZDb65ohgTU8XIB2rYqUScwyYQ3l8NuJIU+P9yNHff4SNKvWYiKKJnsqKfN6CQmq2nNLdjjLnZo7iK2W5Ggt91tl2RYWhyHD/VTUwlykDBBa2PjOS3eb8VNvpyB4jxB/otBVca07iPZuH9wRyYF4wFLAADDFopzzhoRB+1JlMNi+WCHjDyabonEJSfHP7ca5leKS+9GUnqDVtP/MdKBzyYSErz/91gUJdA=', 'tag': 'WtbTx/9/G+iovCNxcISDpw=='}
<Response [200]> {'ok': 'ok'}


# Stop

In [1]:
import json

aad = {
      'sensor_id': 1,
      'transmission_timestamp': 'datetime.now(tz_wib).isoformat()',
      'encrypted_session_key': 'session_key',
  }

aad_json = json.dumps(aad)
aad_bytes = aad_json.encode('utf-8')
print(aad_json)
print(aad_bytes)
print(type(aad_json))
print(type(aad_bytes))
# print(aad.encode('utf-8'))

{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}
b'{"sensor_id": 1, "transmission_timestamp": "datetime.now(tz_wib).isoformat()", "encrypted_session_key": "session_key"}'
<class 'str'>
<class 'bytes'>


In [ ]:
import base64
import json
random_bytes = b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'
print({'random_bytes': random_bytes})
print(type(random_bytes))
A = base64.b64encode(random_bytes)
B = A.decode('utf-8')
print({'random_bytes': A})
print(type(A))
print({'random_bytes': B})
print(type(B))
myjson= json.dumps({'random_bytes' : 1})
print(myjson)
print(type(myjson))

{'random_bytes': b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'}
<class 'bytes'>
{'random_bytes': b'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'bytes'>
{'random_bytes': 'RIW0n6uJRZZDAuWcYksQBbojMUv39ZbYHDRRthUOC0I='}
<class 'str'>
{"random_bytes": 1}
<class 'str'>


In [ ]:
mydict = {'random_bytes': B}
print(type(mydict))

<class 'dict'>


In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

session_key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(session_key)

aad = {
    'mode': ...,
    'algorithm': ...,
    'sensor_id': ...,
    'timestamp': ...,
    'sequence_number': ...,
    'encrypted_session_key': ...,
    # 'nonce': ...,
    # 'ciphertext': ...,
    # 'tag': ...
}

ciphertext_with_tag = aesgcm.encrypt(nonce, plaintext, aad)
ciphertext = ciphertext_with_tag[:-16]
tag = ciphertext_with_tag[-16:]

In [ ]:
import os
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ==========================================
# 1. INSILIALISASI KUNCI ( Dilakukan sekali )
# ==========================================
# GCM membutuhkan kunci simetris minimal 128-bit (16 bytes)
# Kunci ini harus sama dan rahasia antara Sensor dan Server
session_key = AESGCM.generate_key(bit_length=128)
aesgcm = AESGCM(session_key)


# ==========================================
# 2. SISI SENSOR (PROSES ENKRIPSI & KIRIM)
# ==========================================
print("--- SISI SENSOR ---")

# a. Dapatkan Data utama dan Timestamp didapat
timestamp_didapat = datetime.now().isoformat()
nilai_sensor = "24.5 C"

# Gabungkan data utama ke dalam Plaintext (untuk dienkripsi)
plaintext = f"{nilai_sensor}|{timestamp_didapat}".encode('utf-8')

# b. Buat Timestamp Kirim dan Metadata untuk AAD (tidak dienkripsi)
timestamp_kirim = datetime.now().isoformat()
sensor_id = "SENSOR-NODE-01"

# Gabungkan metadata menjadi AAD (Additional Authenticated Data)
aad = f"{sensor_id}|{timestamp_kirim}".encode('utf-8')

# c. Buat IV (Initialization Vector)
# Sesuai rekomendasi NIST, gunakan ukuran tepat 96-bit (12 bytes)
iv = os.urandom(12)

# d. Proses Enkripsi menggunakan AES-GCM
# Fungsi ini otomatis menghasilkan Ciphertext + Authentication Tag jadi satu kesatuan
ciphertext_with_tag = aesgcm.encrypt(iv, plaintext, aad)

print(f"IV (Hex)         : {iv.hex()}")
print(f"AAD Terbuka      : {aad.decode('utf-8')}")
print(f"Ciphertext (Hex) : {ciphertext_with_tag.hex()}\n")

# [ Paket data ini kemudian dikirim melalui jaringan (misal via MQTT/HTTP) ]
# Komponen yang dikirim: iv, aad, dan ciphertext_with_tag



# ==========================================
# 3. SISI SERVER (PROSES DEKRIPSI & VERIFIKASI)
# ==========================================
print("--- SISI SERVER (MENERIMA PAKET) ---")

# Server menerima komponen paket data
paket_iv = iv
paket_aad = aad
paket_crypto = ciphertext_with_tag

try:
    # Server melakukan dekripsi sekaligus verifikasi AAD secara otomatis
    decrypted_data = aesgcm.decrypt(paket_iv, paket_crypto, paket_aad)

    # Jika lolos verifikasi, server membaca data
    # Membongkar AAD (Metadata)
    sensor_id_rec, time_kirim_rec = paket_aad.decode('utf-8').split('|')
    # Membongkar Plaintext (Data Inti)
    nilai_sensor_rec, time_didapat_rec = decrypted_data.decode('utf-8').split('|')

    print("✅ VERIFIKASI BERHASIL: Paket Otentik dan Tidak Dimanipulasi!")
    print(f"Metadata Server -> ID: {sensor_id_rec} | Dikirim: {time_kirim_rec}")
    print(f"Data Sensor     -> Nilai: {nilai_sensor_rec} | Dibaca: {time_didapat_rec}")

except Exception as e:
    # Jika peretas mengubah satu bit saja pada Ciphertext atau AAD (waktu kirim),
    # GCM akan mendeteksinya dan melemparkan error (FAIL).
    print("❌ VERIFIKASI GAGAL: Paket telah dimanipulasi atau kunci salah!")

--- SISI SENSOR ---
IV (Hex)         : 56fc69a043eb2854f549b17e
AAD Terbuka      : SENSOR-NODE-01|2026-05-21T15:50:02.923002
Ciphertext (Hex) : 60bf5c02dc34703adc582f5764e983fc16675b50fc98292ce3ebf8987586bf81c5b8a2bd832056cc561a4b8aaf84680ec6

--- SISI SERVER (MENERIMA PAKET) ---
✅ VERIFIKASI BERHASIL: Paket Otentik dan Tidak Dimanipulasi!
Metadata Server -> ID: SENSOR-NODE-01 | Dikirim: 2026-05-21T15:50:02.923002
Data Sensor     -> Nilai: 24.5 C | Dibaca: 2026-05-21T15:50:02.922861


In [ ]:
print(ciphertext_with_tag)

b'`\xbf\\\x02\xdc4p:\xdcX/Wd\xe9\x83\xfc\x16g[P\xfc\x98),\xe3\xeb\xf8\x98u\x86\xbf\x81\xc5\xb8\xa2\xbd\x83 V\xccV\x1aK\x8a\xaf\x84h\x0e\xc6'


In [ ]:
import os

def generate_aes_key():
    return os.urandom(32)  # 256-bit

print(generate_aes_key())

b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'


In [ ]:
import os
import base64

def generate_aes_key():
    return os.urandom(16)  # 256-bit

iv = generate_aes_key()
iv_64 = base64.b64encode(iv)
json_iv = iv_64.decode("utf-8")
print(iv)
print(iv_64)
print(json_iv)

b"\n\x08\xbf\x00\x93\x83d\xd5\x174\x1a\xa1'\x1e\xc8R"
b'Cgi/AJODZNUXNBqhJx7IUg=='
Cgi/AJODZNUXNBqhJx7IUg==


In [ ]:
# from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# from cryptography.hazmat.primitives import padding
import json

data = {
        'timestamp': '2026-05-17T01:08:00Z',
        'sensor_id': '01',
        'readings': {
            'temperature': 26,
            'humidity': 93
        }
}

plaintext = json.dumps(data)
print(plaintext)

{"timestamp": "2026-05-17T01:08:00Z", "sensor_id": "01", "readings": {"temperature": 26, "humidity": 93}}


In [ ]:
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.serialization import load_pem_public_key

def rsa_encrypt(aes_key: bytes, public_key_pem) -> bytes:
    # public_key_pem = get_server_public_key()
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    ciphertext = public_key.encrypt(
        aes_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return ciphertext

print(rsa_encrypt(b'D\x85\xb4\x9f\xab\x89E\x96C\x02\xe5\x9cbK\x10\x05\xba#1K\xf7\xf5\x96\xd8\x1c4Q\xb6\x15\x0e\x0bB'
,'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAt8naqge5WFHlloyPDMwN\n3JZoV26EPzfhdC7MkYD6MLwvOesXn2lM9PBNt8kjEyb8lBnAyWYmWA/HJ0GiP07l\n++NoIYSCJIJttQhzD+LDzUjWhf5poOWDwE7GZlAQ2Aqj+ffBdOI7D2RBN+4YlT2t\ne0UmNJ7Y8AvukGHDMW8TCN8Arp6Rx/wqI1y61AH6IfQ+igvdMjTmKGlXuMNnu+bO\nLg8ig7jy4FtLNHRxnL/LTMnDH7+rXV72cT1Rw8yAWwhOQS6D8IYsWhpJ8YnC7ghw\nHdNuUjm6SfpZKHuJyz/yg6siM3TS7xlAXVT2yktBw5YnN0Da7duOMqkODZupMjGw\nrwIDAQAB\n-----END PUBLIC KEY-----\n'))

b'\x06.\xc5B"\xcf=t\x19\xaf\x8bb\xcb5\xbc\x8c\x80\xf6\x06\xa2\xe1}$p\x96\x079\x14\tA\xa2\x1b\xf0\x1a\xb4\xe7\',\xedG>2\xe2E"m\xe8\x85\x94\xb7\xce\x18\xa3\xf3s\x93\xe7\t\xff,\x19;\x9c\x15Ih\x04.}]\x97w\x87G\x19>\xd7\xfe\xdf\x7f\xe4&v\x90\xd7\x87\r\x0b\x8a\xe0\x99&\xd2pN\xee\xba\x9f\\,y\x90\n\x8f\xa3\x01\x07\xcbU\x90_b\xfe^\xeca\xe0\xa3%\x11\xdd\xecX\xa0u\xc3\xce\x93)(\xde\x1c\x1f\x87\xe4\x1e\xa7\x97\xb2\x83\xbaT\xd7\'L\x08Y t7\xb6\x8e\x02`\x0f\x9b\n\x87\x7f\xedf\x1a\xf9#\r\x8e+\xb1n\xdf\xad\xa1\t\xe6\xfa\xb7\xddz`\x0f\x01O\x16\x11*\xc2p94g\t\xe9\x80\xc0\xe2\x05\xe9\xe1c\xdc\x84+\xb1\x89}\xeey\xa4\xe5\x0f\x9dg\x88\xcb\x8dr%\n\x96\x7f\xd1\xcc\x04\xf6\x1a^\xed:\xc5\xfd\xac/\xdb\xc2\xbaP\xd7nC^\x14\x98\xd3\xea\xf5\xbe4y9\xbe2TI\xba\x87='


In [ ]:
sensor_data = {}

def generate_sensor_data():
    # global sensor_data
    sensor_data = {42}
    print(sensor_data)

generate_sensor_data()
print(sensor_data)

{42}
{}


In [ ]:
def generate_sensor_data():
  state.sensor_data = {42}

class State:
  sensor_data = None

state = State()



generate_sensor_data()
print(state.sensor_data)

{42}
